# Checkpoint fidelity test (two-tower)

Must-pass check: in-memory model right after training vs model reloaded from `.keras` on the **same** eval examples.

**Re-run the train cell** after updating `two_tower_train.py` so the checkpoint is saved with all layers built (old `fidelity_test.keras` files only contain `item_base_embeddings`).

Pass if score vectors and top-k rankings match within tight tolerances. Masked query-app scores use `-inf` and are treated as valid, not NaN.

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd

from steam_review_ml.constants import PROJECT_RANDOM_SEED
from steam_review_ml.recommender.contrastive_examples import build_contrastive_examples
from steam_review_ml.recommender.retrieve import ContentRetriever
from steam_review_ml.recommender.two_tower_train import (
    TwoTowerTrainConfig,
    build_tower_contrastive_rows,
    item_init_matrix_from_catalog,
    load_hub_settings,
    train_two_tower,
)
from steam_review_ml.recommender.two_tower_score import (
    load_two_tower_model,
    make_two_tower_score_fn,
    precompute_catalog_item_vectors,
)
from steam_review_ml.evaluation.retrieval_offline_eval import (
    load_eval_examples_from_parquet,
    _rank_rows,
    hit_rate_at_k,
    recall_at_k,
)

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
ARTIFACT_DIR = REPO_ROOT / "artifacts/recs"
MODEL_PATH = REPO_ROOT / "artifacts/recs/towers/val_dev_12k_v1/fidelity_test.keras"
EVAL_EXAMPLES_PATH = REPO_ROOT / "artifacts/recs/eval_cache/val_dev_12k_v1/eval_examples.parquet"

train_cfg = TwoTowerTrainConfig(
    training_mode="full",
    batch_size=64,
    proj_dim=64,
    temperature=0.07,
    epochs=2,
    smoke_epochs=2,
    early_stopping_patience=1,
    tower_seed=2026,
    adam_lr=1e-3,
)

MAX_TRAIN_EXAMPLES = 50_000
MAX_VAL_EXAMPLES = 2_000
MIN_REVIEW_CHARS = 30
MAX_TRAIN_ROWS_PER_USER = 5
RANDOM_SEED = PROJECT_RANDOM_SEED

N_EVAL_EXAMPLES = 200
K_FINAL = 10
K_RETRIEVAL = 100

/home/ryanr/miniconda3/envs/tf_condaforge/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
retriever = ContentRetriever(artifact_dir=ARTIFACT_DIR, repo_root=REPO_ROOT)
hub_url, hub_max_chars = load_hub_settings(retriever)
item_init = item_init_matrix_from_catalog(retriever)

train_examples, _ = build_contrastive_examples(
    repo_root=REPO_ROOT,
    split="train",
    max_examples=MAX_TRAIN_EXAMPLES,
    min_review_chars=MIN_REVIEW_CHARS,
    max_train_rows_per_user=MAX_TRAIN_ROWS_PER_USER,
    random_seed=RANDOM_SEED,
    artifact_dir=ARTIFACT_DIR,
    verbose=False,
)
val_examples, _ = build_contrastive_examples(
    repo_root=REPO_ROOT,
    split="val",
    max_examples=MAX_VAL_EXAMPLES,
    min_review_chars=MIN_REVIEW_CHARS,
    max_train_rows_per_user=MAX_TRAIN_ROWS_PER_USER,
    random_seed=RANDOM_SEED + 1,
    artifact_dir=ARTIFACT_DIR,
    verbose=False,
)

train_cap = min(MAX_TRAIN_EXAMPLES, len(train_examples))
val_cap = min(MAX_VAL_EXAMPLES, len(val_examples))
train_payload = build_tower_contrastive_rows(
    train_examples, retriever, max_examples=train_cap, max_chars=hub_max_chars
)
val_payload = build_tower_contrastive_rows(
    val_examples, retriever, max_examples=val_cap, max_chars=hub_max_chars
)

model_in_mem, history = train_two_tower(
    train_payload,
    val_payload,
    train_cfg,
    hub_url=hub_url,
    item_init_matrix=item_init,
)

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
if not model_in_mem.built:
    model_in_mem.build()
print("n_weights before save:", len(model_in_mem.weights))
model_in_mem.save(MODEL_PATH)
print("Saved:", MODEL_PATH)

In [ ]:
model_loaded = load_two_tower_model(
    MODEL_PATH,
    hub_url=hub_url,
    n_items=len(retriever.app_ids),
    embed_dim=int(retriever.embedding_matrix.shape[1]),
)
print("Loaded model OK")

Loaded model OK


## Reload diagnostics (before eval scoring)

Run in order: **weights** → **single forward pass** → **full catalog item vectors**. If any step fails, fix save/load before interpreting the 200-example score comparison below.

In [ ]:
import tensorflow as tf

WEIGHT_ATOL = 1e-6
PROBE_TEXT = "great game very fun"


def report_weight_diff(model_a, model_b, *, atol: float = WEIGHT_ATOL) -> tuple[pd.DataFrame, bool]:
    weights_a = list(model_a.weights)
    weights_b = list(model_b.weights)
    rows: list[dict] = []
    all_ok = True
    if len(weights_a) != len(weights_b):
        rows.append(
            {
                "name": "(count mismatch)",
                "max_abs_diff": np.nan,
                "ok": False,
                "note": f"in_mem={len(weights_a)} loaded={len(weights_b)}",
            }
        )
        return pd.DataFrame(rows), False
    for wa, wb in zip(weights_a, weights_b):
        if wa.shape != wb.shape:
            rows.append(
                {
                    "name": wa.name,
                    "max_abs_diff": np.nan,
                    "ok": False,
                    "note": f"shape {wa.shape} vs {wb.shape}",
                }
            )
            all_ok = False
            continue
        diff = float(np.max(np.abs(wa.numpy() - wb.numpy())))
        ok = diff < atol
        rows.append({"name": wa.name, "max_abs_diff": diff, "ok": ok, "note": ""})
        all_ok = all_ok and ok
    return pd.DataFrame(rows), all_ok


def scores_valid(scores: np.ndarray) -> bool:
    """True when finite or intentionally masked with -inf (query app)."""
    s = np.asarray(scores)
    return bool(np.all(np.isfinite(s) | (s == -np.inf)))


# 1) Weights only
weight_df, weights_ok = report_weight_diff(model_in_mem, model_loaded)
print("=== 1) Trainable weights ===")
display(weight_df)
print("weights_ok:", weights_ok)

# 2) One forward pass (no full catalog)
probe_text_batch = tf.constant([PROBE_TEXT], dtype=tf.string)
probe_row_batch = tf.constant([0, 1, 2], dtype=tf.int32)
forward_rows: list[dict] = []
for label, model in [("in_mem", model_in_mem), ("loaded", model_loaded)]:
    user_vec = model.encode_user(probe_text_batch).numpy()
    item_vec = model.encode_item_by_row_ids(probe_row_batch).numpy()
    forward_rows.append(
        {
            "model": label,
            "user_finite": bool(np.isfinite(user_vec).all()),
            "item_finite": bool(np.isfinite(item_vec).all()),
            "user_std": float(np.nanstd(user_vec)),
            "item_std": float(np.nanstd(item_vec)),
        }
    )
forward_df = pd.DataFrame(forward_rows)
print("\n=== 2) Single forward pass ===")
display(forward_df)
forward_ok = bool(forward_df[["user_finite", "item_finite"]].all().all())
print("forward_ok:", forward_ok)

# 3) Full catalog item vectors (what make_two_tower_score_fn precomputes)
catalog_rows: list[dict] = []
n_items = len(retriever.app_ids)
for label, model in [("in_mem", model_in_mem), ("loaded", model_loaded)]:
    item_vectors = precompute_catalog_item_vectors(model, n_items, batch_size=256)
    finite_frac = float(np.isfinite(item_vectors).mean())
    catalog_rows.append(
        {
            "model": label,
            "shape": str(item_vectors.shape),
            "finite_frac": finite_frac,
            "std": float(np.nanstd(item_vectors)),
        }
    )
catalog_df = pd.DataFrame(catalog_rows)
print("\n=== 3) Catalog item vectors ===")
display(catalog_df)
catalog_ok = bool((catalog_df["finite_frac"] >= 1.0).all())
print("catalog_ok:", catalog_ok)

reload_diagnostics_ok = weights_ok and forward_ok and catalog_ok
print("\nreload_diagnostics_ok:", reload_diagnostics_ok)

=== 1) Trainable weights ===


,name,max_abs_diff,ok,note
0,item_base_embeddings,0.0,True,
1,kernel,0.0,True,
2,bias,0.0,True,
3,kernel,0.0,True,
4,bias,0.0,True,


weights_ok: True

=== 2) Single forward pass ===


,model,user_finite,item_finite,user_std,item_std
0,in_mem,True,True,0.241434,0.353624
1,loaded,True,True,0.241434,0.353624


forward_ok: True

=== 3) Catalog item vectors ===


,model,shape,finite_frac,std
0,in_mem,"(315, 64)",1.0,0.378875
1,loaded,"(315, 64)",1.0,0.378875


catalog_ok: True

reload_diagnostics_ok: True


In [ ]:
score_in_mem = make_two_tower_score_fn(
    model_in_mem,
    retriever,
    max_chars=hub_max_chars,
    catalog_item_batch=256,
    mask_query_app=True,
)
score_loaded = make_two_tower_score_fn(
    model_loaded,
    retriever,
    max_chars=hub_max_chars,
    catalog_item_batch=256,
    mask_query_app=True,
)

examples = load_eval_examples_from_parquet(EVAL_EXAMPLES_PATH)[:N_EVAL_EXAMPLES]
app_ids = np.asarray(retriever.app_ids)
print(f"Eval sample: {len(examples)} examples")

Eval sample: 200 examples


In [ ]:
def cosine(a: np.ndarray, b: np.ndarray, eps: float = 1e-12) -> float:
    na = np.linalg.norm(a)
    nb = np.linalg.norm(b)
    if na < eps or nb < eps:
        return 0.0
    return float(np.dot(a, b) / (na * nb))

rows = []
for i, ex in enumerate(examples):
    s1 = np.asarray(score_in_mem(ex), dtype=np.float64)
    s2 = np.asarray(score_loaded(ex), dtype=np.float64)
    s1_ok = scores_valid(s1)
    s2_ok = scores_valid(s2)
    top10_1 = _rank_rows(s1)[:K_FINAL]
    top10_2 = _rank_rows(s2)[:K_FINAL]
    positives = {int(x) for x in ex["validation_positive_app_ids"]}
    comparable = np.isfinite(s1) & np.isfinite(s2)
    if s1_ok and s2_ok and comparable.any():
        max_abs_diff = float(np.max(np.abs(s1[comparable] - s2[comparable])))
        score_cosine = cosine(s1[comparable], s2[comparable])
    else:
        max_abs_diff = np.nan
        score_cosine = np.nan
    rows.append(
        {
            "ex_idx": i,
            "s1_ok": s1_ok,
            "s2_ok": s2_ok,
            "max_abs_diff": max_abs_diff,
            "cosine": score_cosine,
            "top10_exact": int(np.array_equal(top10_1, top10_2)),
            "top10_overlap": float(len(set(top10_1.tolist()) & set(top10_2.tolist())) / K_FINAL),
            "hit10_in_mem": hit_rate_at_k(top10_1, positives, K_FINAL, app_ids),
            "hit10_loaded": hit_rate_at_k(top10_2, positives, K_FINAL, app_ids),
            "recall100_in_mem": recall_at_k(_rank_rows(s1)[:K_RETRIEVAL], positives, K_RETRIEVAL, app_ids),
            "recall100_loaded": recall_at_k(_rank_rows(s2)[:K_RETRIEVAL], positives, K_RETRIEVAL, app_ids),
        }
    )

df = pd.DataFrame(rows)
valid_pairs = int((df["s1_ok"] & df["s2_ok"]).sum())
summary = {
    "n_examples": len(df),
    "valid_score_pairs": valid_pairs,
    "s1_ok_rate": float(df["s1_ok"].mean()),
    "s2_ok_rate": float(df["s2_ok"].mean()),
    "n_weights_in_mem": len(model_in_mem.weights),
    "n_weights_loaded": len(model_loaded.weights),
    "max_abs_diff_max": float(df["max_abs_diff"].max()),
    "max_abs_diff_p95": float(df["max_abs_diff"].quantile(0.95)),
    "cosine_mean": float(df["cosine"].mean()),
    "top10_exact_rate": float(df["top10_exact"].mean()),
    "top10_overlap_mean": float(df["top10_overlap"].mean()),
    "hit10_delta_abs": float(abs(df["hit10_in_mem"].mean() - df["hit10_loaded"].mean())),
    "recall100_delta_abs": float(abs(df["recall100_in_mem"].mean() - df["recall100_loaded"].mean())),
}
summary

{'n_examples': 200,
 'valid_score_pairs': 200,
 's1_ok_rate': 1.0,
 's2_ok_rate': 1.0,
 'n_weights_in_mem': 5,
 'n_weights_loaded': 5,
 'max_abs_diff_max': 0.0,
 'max_abs_diff_p95': 0.0,
 'cosine_mean': 1.0,
 'top10_exact_rate': 1.0,
 'top10_overlap_mean': 1.0,
 'hit10_delta_abs': 0.0,
 'recall100_delta_abs': 0.0}

In [ ]:
passed = (
    reload_diagnostics_ok
    and summary["n_weights_in_mem"] == summary["n_weights_loaded"]
    and summary["valid_score_pairs"] == summary["n_examples"]
    and summary["max_abs_diff_max"] < 1e-5
    and summary["cosine_mean"] > 0.99999
    and summary["top10_exact_rate"] > 0.99
    and summary["hit10_delta_abs"] < 1e-3
    and summary["recall100_delta_abs"] < 1e-3
)

print("Fidelity test:", "PASS" if passed else "FAIL")
print(json.dumps(summary, indent=2))
df.sort_values("max_abs_diff", ascending=False).head(10)

Fidelity test: PASS
{
  "n_examples": 200,
  "valid_score_pairs": 200,
  "s1_ok_rate": 1.0,
  "s2_ok_rate": 1.0,
  "n_weights_in_mem": 5,
  "n_weights_loaded": 5,
  "max_abs_diff_max": 0.0,
  "max_abs_diff_p95": 0.0,
  "cosine_mean": 1.0,
  "top10_exact_rate": 1.0,
  "top10_overlap_mean": 1.0,
  "hit10_delta_abs": 0.0,
  "recall100_delta_abs": 0.0
}


,ex_idx,s1_ok,s2_ok,max_abs_diff,cosine,top10_exact,top10_overlap,hit10_in_mem,hit10_loaded,recall100_in_mem,recall100_loaded
0,0,True,True,0.0,1.0,1,1.0,0.0,0.0,1.0,1.0
1,1,True,True,0.0,1.0,1,1.0,0.0,0.0,1.0,1.0
2,2,True,True,0.0,1.0,1,1.0,0.0,0.0,1.0,1.0
3,3,True,True,0.0,1.0,1,1.0,0.0,0.0,1.0,1.0
4,4,True,True,0.0,1.0,1,1.0,0.0,0.0,1.0,1.0
5,5,True,True,0.0,1.0,1,1.0,0.0,0.0,0.0,0.0
6,6,True,True,0.0,1.0,1,1.0,0.0,0.0,1.0,1.0
7,7,True,True,0.0,1.0,1,1.0,0.0,0.0,0.0,0.0
8,8,True,True,0.0,1.0,1,1.0,1.0,1.0,1.0,1.0
9,9,True,True,0.0,1.0,1,1.0,0.0,0.0,1.0,1.0


In [ ]:
GO_NO_GO_THRESHOLDS = {
    "max_abs_diff_max": 1e-5,
    "cosine_mean": 0.99999,
    "top10_exact_rate": 0.99,
    "hit10_delta_abs": 1e-3,
    "recall100_delta_abs": 1e-3,
}

checks = {
    "reload_diagnostics_ok": bool(reload_diagnostics_ok),
    "weights_count_match": summary["n_weights_in_mem"] == summary["n_weights_loaded"],
    "all_scores_valid": summary["valid_score_pairs"] == summary["n_examples"],
    "max_abs_diff_ok": summary["max_abs_diff_max"] < GO_NO_GO_THRESHOLDS["max_abs_diff_max"],
    "cosine_ok": summary["cosine_mean"] > GO_NO_GO_THRESHOLDS["cosine_mean"],
    "top10_exact_ok": summary["top10_exact_rate"] > GO_NO_GO_THRESHOLDS["top10_exact_rate"],
    "hit10_delta_ok": summary["hit10_delta_abs"] < GO_NO_GO_THRESHOLDS["hit10_delta_abs"],
    "recall100_delta_ok": summary["recall100_delta_abs"] < GO_NO_GO_THRESHOLDS["recall100_delta_abs"],
}

go_no_go = all(checks.values())

print("=== Fidelity go/no-go ===")
for key, ok in checks.items():
    print(f"{key:24s}: {'PASS' if ok else 'FAIL'}")
print(f"\nFINAL: {'GO' if go_no_go else 'NO-GO'}")

pd.DataFrame(
    [
        {
            "metric": "weights",
            "in_mem": summary["n_weights_in_mem"],
            "loaded": summary["n_weights_loaded"],
            "delta_abs": abs(summary["n_weights_in_mem"] - summary["n_weights_loaded"]),
        },
        {
            "metric": "score_valid_pairs",
            "in_mem": summary["n_examples"],
            "loaded": summary["valid_score_pairs"],
            "delta_abs": abs(summary["n_examples"] - summary["valid_score_pairs"]),
        },
        {
            "metric": "hit10",
            "in_mem": float(df["hit10_in_mem"].mean()),
            "loaded": float(df["hit10_loaded"].mean()),
            "delta_abs": summary["hit10_delta_abs"],
        },
        {
            "metric": "recall100",
            "in_mem": float(df["recall100_in_mem"].mean()),
            "loaded": float(df["recall100_loaded"].mean()),
            "delta_abs": summary["recall100_delta_abs"],
        },
    ]
)

=== Fidelity go/no-go ===
reload_diagnostics_ok   : PASS
weights_count_match     : PASS
all_scores_valid        : PASS
max_abs_diff_ok         : PASS
cosine_ok               : PASS
top10_exact_ok          : PASS
hit10_delta_ok          : PASS
recall100_delta_ok      : PASS

FINAL: GO


,metric,in_mem,loaded,delta_abs
0,weights,5.000000,5.000000,0.0
1,score_valid_pairs,200.000000,200.000000,0.0
2,hit10,0.070000,0.070000,0.0
3,recall100,0.460208,0.460208,0.0
